# 🧹 Module 2: Data Cleaning & Feature Engineering

In this notebook, we perform data preprocessing and cleaning steps:
1. Remove duplicate rows
2. Remove missing `CustomerID` records
3. Remove cancelled invoices (where `InvoiceNo` starts with 'C')
4. Filter out negative or zero `Quantity` and `UnitPrice`
5. Convert `InvoiceDate` to datetime format
6. Create `TotalPrice = Quantity × UnitPrice`
7. Extract temporal features: `Year`, `Month`, `Day`, `Weekday`, `Hour`
8. Save cleaned dataset to `data/processed/cleaned_retail_data.csv`

---

In [ ]:
import os
import pandas as pd
import numpy as np

raw_path = os.path.join("..", "data", "raw", "Online Retail.xlsx")
df = pd.read_excel(raw_path)
print(f"Initial Raw Dataset Shape: {df.shape}")

### Step 1: Remove Duplicate Rows

In [ ]:
dup_count = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dup_count:,} duplicate rows. New Shape: {df.shape}")

### Step 2: Remove Missing CustomerID

In [ ]:
missing_cust = df["CustomerID"].isnull().sum()
df = df.dropna(subset=["CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
print(f"Removed {missing_cust:,} records with missing CustomerID. New Shape: {df.shape}")

### Step 3: Remove Cancelled Orders (InvoiceNo starting with 'C')

In [ ]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
cancelled = df["InvoiceNo"].str.upper().str.startswith("C").sum()
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]
print(f"Removed {cancelled:,} cancelled order rows. New Shape: {df.shape}")

### Step 4: Remove Quantity <= 0 & UnitPrice <= 0

In [ ]:
invalid_qty = (df["Quantity"] <= 0).sum()
invalid_price = (df["UnitPrice"] <= 0).sum()

df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print(f"Removed {invalid_qty} non-positive quantity rows and {invalid_price} zero-price rows.")
print(f"Cleaned Valid Dataset Shape: {df.shape}")

### Step 5: Feature Engineering (TotalPrice & Datetime Features)

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# Extract temporal components
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["Weekday"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

df.head()

### Step 6: Export Cleaned Dataset to CSV

In [ ]:
out_dir = os.path.join("..", "data", "processed")
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "cleaned_retail_data.csv")

df.to_csv(out_file, index=False)
print(f"Cleaned dataset saved successfully to: {out_file}")